# Compute Metrics Demo

This notebook demonstrates how to use the `compute_presentation_metrics.py` script to:

1. Load per-frame prediction data
2. Compute evaluation metrics (AUROC, AUPRC) with bootstrap confidence intervals
3. Generate ROC/PR curves and score distribution plots
4. Compute operational metrics (precision@k, recall@FPR)

## For Chemists and Scientists

This tool helps you evaluate how well the anomaly detection model identifies unusual molecular conformations or dynamics in your MD simulations. The metrics tell you:

- **AUROC (Area Under ROC Curve)**: How well the model distinguishes anomalous frames from normal ones. Values closer to 1.0 are better; 0.5 means random guessing.

- **AUPRC (Area Under Precision-Recall Curve)**: Especially useful when anomalies are rare. Measures how precisely the model identifies true anomalies.

- **Precision@k**: Among the top k% of frames flagged as anomalous, what fraction are truly anomalous?

- **Recall@FPR**: At a fixed false positive rate, how many true anomalies do we catch?

## 1. Setup and Imports

In [ ]:
import sys
import subprocess
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Ensure we're in the repository root
repo_root = Path.cwd()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent

print(f"Repository root: {repo_root}")

## 2. Examine Sample Predictions Data

Let's look at the format of the predictions file. The script expects:
- `frame`: Frame index
- `y_true`: Ground truth label (0 = normal, 1 = anomaly)
- `y_score`: Anomaly score from the model
- `run_id` (optional): Identifier for different runs/methods

In [ ]:
# Load sample predictions
sample_path = repo_root / 'tests' / 'sample_predictions.csv'
df = pd.read_csv(sample_path)

print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst 10 rows:")
display(df.head(10))

In [ ]:
# Summary statistics
print("Class distribution:")
print(df['y_true'].value_counts())
print(f"\nAnomaly rate: {df['y_true'].mean():.1%}")

if 'run_id' in df.columns:
    print(f"\nRuns: {df['run_id'].unique().tolist()}")

## 3. Run the Metrics Script

Now let's run the metrics computation script. You can run it from:
1. This notebook (using subprocess)
2. Command line directly

### Command Line Usage

```bash
# Using default settings (auto-detect predictions)
python tools/compute_presentation_metrics.py

# Using specific predictions file
python tools/compute_presentation_metrics.py \
    --predictions outputs/predictions.csv \
    --out-dir outputs/summary \
    --bootstrap 2000 \
    --seed 42
```

In [ ]:
# Run the metrics script on sample data
output_dir = repo_root / 'outputs' / 'demo_metrics'
output_dir.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    str(repo_root / 'tools' / 'compute_presentation_metrics.py'),
    '--predictions', str(sample_path),
    '--out-dir', str(output_dir),
    '--bootstrap', '500',  # Fewer resamples for demo speed
    '--seed', '42'
]

print("Running command:")
print(' '.join(cmd))
print()

result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(repo_root))

if result.returncode == 0:
    print("✓ Script completed successfully!")
    print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
else:
    print("✗ Script failed:")
    print(result.stderr)

## 4. View Results

### 4.1 Metrics Summary

In [ ]:
# Load and display metrics
metrics_path = output_dir / 'metrics_summary.csv'
if metrics_path.exists():
    metrics_df = pd.read_csv(metrics_path)
    display(metrics_df)
    
    # Print formatted summary
    row = metrics_df.iloc[0]
    print("\n" + "="*60)
    print("EVALUATION METRICS SUMMARY")
    print("="*60)
    print(f"\nSamples: {int(row.get('n_samples', 0))}")
    print(f"Positive: {int(row.get('n_positive', 0))}")
    print(f"Negative: {int(row.get('n_negative', 0))}")
    print(f"\nAUROC: {row['auroc']:.4f} (95% CI: {row['auroc_lo']:.4f} – {row['auroc_hi']:.4f})")
    print(f"AUPRC: {row['auprc']:.4f} (95% CI: {row['auprc_lo']:.4f} – {row['auprc_hi']:.4f})")
    
    # Operational metrics
    print("\nOperational Metrics:")
    for col in metrics_df.columns:
        if 'precision_at' in col or 'recall_at' in col:
            print(f"  {col}: {row[col]:.4f}")
else:
    print("Metrics file not found. Run the script first.")

### 4.2 ROC Curve

The ROC curve shows the trade-off between True Positive Rate (catching anomalies) and False Positive Rate (false alarms).

In [ ]:
roc_path = output_dir / 'predictions_roc.png'
if roc_path.exists():
    display(Image(filename=str(roc_path)))
else:
    print("ROC curve not found. Run the script first.")

### 4.3 Precision-Recall Curve

The PR curve is especially useful when anomalies are rare. It shows how precision (fraction of flagged frames that are truly anomalous) changes with recall (fraction of anomalies caught).

In [ ]:
pr_path = output_dir / 'predictions_pr.png'
if pr_path.exists():
    display(Image(filename=str(pr_path)))
else:
    print("PR curve not found. Run the script first.")

### 4.4 Score Distributions

This plot shows the distribution of anomaly scores for normal vs. anomalous frames. Good separation between the distributions means the model can distinguish anomalies well.

In [ ]:
dist_path = output_dir / 'score_distributions.png'
if dist_path.exists():
    display(Image(filename=str(dist_path)))
else:
    print("Score distributions not found. Run the script first.")

## 5. Per-Run Analysis (if applicable)

If your predictions file contains multiple runs (e.g., different methods or cross-validation folds), the script computes per-run metrics.

In [ ]:
per_run_path = output_dir / 'metrics_summary_per_run.csv'
if per_run_path.exists():
    per_run_df = pd.read_csv(per_run_path)
    print("Per-Run Metrics:")
    display(per_run_df)
    
    # Summary statistics across runs
    print("\nSummary across runs:")
    print(f"  AUROC: {per_run_df['auroc'].mean():.4f} ± {per_run_df['auroc'].std():.4f}")
    print(f"  AUPRC: {per_run_df['auprc'].mean():.4f} ± {per_run_df['auprc'].std():.4f}")
else:
    print("No per-run metrics found (single run in data).")

## 6. Interpretation Guide for Chemists

### What do these metrics mean for your analysis?

1. **AUROC > 0.8**: The model has good ability to distinguish anomalous frames from normal ones.

2. **AUROC > 0.9**: Excellent discrimination - the model rarely confuses anomalies with normal behavior.

3. **Precision@1%**: If you only have time to manually inspect the top 1% of flagged frames, what fraction will be true anomalies? Higher is better.

4. **Recall@5%FPR**: If you can tolerate a 5% false alarm rate, what fraction of true anomalies will you catch?

### Practical recommendations:

- **Initial screening**: Focus on Top 1% highest-scoring frames
- **Broader investigation**: Expand to Top 5% for more coverage
- **Threshold selection**: Use the score distribution plot to choose a threshold that balances precision and recall for your needs

### Confidence intervals:

- **Narrow CI**: Stable estimate, you can trust this value
- **Wide CI**: More data might be needed for reliable conclusions
- **Overlapping CIs**: If comparing methods, overlapping CIs suggest no significant difference

## 7. Using Your Own Data

To use this script with your own predictions:

```python
# Create a DataFrame with your predictions
import pandas as pd

df = pd.DataFrame({
    'frame': range(len(your_scores)),
    'y_true': your_labels,  # 0 or 1
    'y_score': your_scores,  # Model scores
    'run_id': 'my_method'  # Optional
})

# Save to CSV
df.to_csv('my_predictions.csv', index=False)

# Run the script
# python tools/compute_presentation_metrics.py --predictions my_predictions.csv
```

In [ ]:
# Cleanup demo outputs (optional)
# import shutil
# shutil.rmtree(output_dir)